# Partial point cloud dataset from ACRONYM

This notebook renders single-view depth observations of ACRONYM objects and stores, per view:

* a noisy **partial point cloud** (ray cast depth image with sensor noise, `helper/rendering.py`, same renderer as `partial_mode: render` in training),
* the object's ACRONYM **grasps** (successful and failed), their two finger **contact points**,
* a **visibility class** per grasp:

| class | meaning |
|---|---|
| `2` | both contact points visible |
| `1` | exactly one contact point visible |
| `0` | no contact point visible |
| `-1` | grasp has fewer than two contacts on the mesh (fingers close on air), not classified |

A contact counts as visible if it lies inside the camera frustum and the line of sight from the camera to the contact is free (first ray hit within `VISIBILITY_TOLERANCE` of the contact). The distance from every contact to the nearest observed point is stored too, so the classes can be recomputed with a stricter "seen in the noisy cloud" criterion.

**Frames.** Everything is stored in the **camera frame** (OpenCV convention: x right, y down, z forward). `camera_pose` maps camera to ACRONYM mesh coordinates (mesh scaled by `object/scale`, not centred). Grasp poses are the raw ACRONYM Panda poses (wrist frame, fingers close along x, finger pads at z = 0.103 to 0.112 m).

The last section uploads the dataset to the Hugging Face Hub with **your** token. It only runs when `UPLOAD = True`.

## 1. Setup

In [ ]:
# On Colab / a fresh environment:
# !pip install -q open3d trimesh h5py scipy datasets huggingface_hub matplotlib
import os, sys, glob, hashlib, json
import numpy as np
import h5py
import trimesh
from scipy.spatial import cKDTree

REPO_ROOT = os.environ.get("DIGNET_ROOT", os.path.abspath(".."))  # notebook lives in DIGNet/notebooks
sys.path.insert(0, REPO_ROOT)
from helper.rendering import DepthRenderer

## 2. Configuration

In [ ]:
ACRONYM_ROOT = os.environ.get("ACRONYM_ROOT", "/path/to/acronym")   # contains grasps/ and meshes/
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", os.path.join(ACRONYM_ROOT, "partial_views_dataset"))

MAX_OBJECTS = int(os.environ.get("MAX_OBJECTS", 0)) or None   # None = all objects
VIEWS_PER_OBJECT = int(os.environ.get("VIEWS_PER_OBJECT", 4))
NUM_POINTS = 1024                    # points per partial cloud (resampled)
MAX_GRASPS_PER_OBJECT = 500          # balanced between successful and failed grasps where possible
VISIBILITY_TOLERANCE = 0.003         # m, line of sight tolerance
SPLITS = {"train": 0.90, "validation": 0.05, "test": 0.05}   # per object, deterministic hash split
NUM_PROC = int(os.environ.get("NUM_PROC", max(1, (os.cpu_count() or 2) // 2)))
SEED = 0

# camera and sensor model, see helper/rendering.py
RENDERER = DepthRenderer(width=160, height=120, fov_deg=58.0, distance_range=(0.35, 0.7), min_elevation=-0.2,
                         sigma_base=0.001, sigma_quad=0.002, dropout=0.02, edge_threshold=0.01, edge_dropout=0.5)

# Panda finger pads in the ACRONYM gripper frame (helper/gripper_control_points/panda_gripper_coords.yml)
FINGER_X = 0.0399                    # inner pad surface at full opening
PAD_Z = (0.1034, 0.1078, 0.112204)   # pad centre to tip
PAD_Y = (-0.008, 0.0, 0.008)
MAX_CLOSING = 2 * FINGER_X

HF_REPO_ID = "your-username/acronym-partial-views"   # dataset repo on the Hub
HF_PRIVATE = True
UPLOAD = False

h5_files = sorted(glob.glob(os.path.join(ACRONYM_ROOT, "grasps", "*.h5")))[:MAX_OBJECTS]
print(f"{len(h5_files)} objects, {VIEWS_PER_OBJECT} views each")

## 3. Helpers

In [ ]:
def load_object(h5_path):
    with h5py.File(h5_path, "r") as f:
        mesh_file = f["object/file"][()].decode("utf-8")
        scale = float(f["object/scale"][()])
        grasps = np.array(f["grasps/transforms"])
        success = np.array(f["grasps/qualities/flex/object_in_gripper"]).astype(np.int8)
    mesh = trimesh.load(os.path.join(ACRONYM_ROOT, mesh_file), force="mesh")
    mesh.apply_scale(scale)
    return mesh, grasps, success, mesh_file, scale


def select_grasps(grasps, success, rng):
    # balanced subset: up to half successful, fill with the other class
    pos, neg = np.flatnonzero(success == 1), np.flatnonzero(success == 0)
    n_pos = min(len(pos), max(MAX_GRASPS_PER_OBJECT // 2, MAX_GRASPS_PER_OBJECT - len(neg)))
    n_neg = min(len(neg), MAX_GRASPS_PER_OBJECT - n_pos)
    idx = np.concatenate([rng.choice(pos, n_pos, replace=False), rng.choice(neg, n_neg, replace=False)])
    return np.sort(idx)


def finger_contacts(scene, grasps):
    # closing rays from both finger pads towards the gripper centre, first hit within the stroke = contact
    import open3d as o3d
    pads = np.array([[s * FINGER_X, y, z, 1.0] for s in (1, -1) for z in PAD_Z for y in PAD_Y])  # [2*R, 4]
    R = len(PAD_Z) * len(PAD_Y)
    origins = np.einsum("gij,pj->gpi", grasps[:, :3, :], pads)                     # [G, 2R, 3]
    directions = -np.sign(pads[:, 0])[None, :, None] * grasps[:, None, :3, 0]      # towards the other finger
    rays = np.concatenate([origins, directions], axis=-1).reshape(-1, 6).astype(np.float32)
    t = scene.cast_rays(o3d.core.Tensor(rays))["t_hit"].numpy().reshape(len(grasps), 2, R)
    t = np.where(t <= MAX_CLOSING, t, np.inf)
    best = t.argmin(axis=-1)                                                         # [G, 2]
    t_best = np.take_along_axis(t, best[..., None], -1)[..., 0]
    o = origins.reshape(len(grasps), 2, R, 3)
    d = directions.reshape(len(grasps), 2, R, 3)
    pick = lambda a: np.take_along_axis(a, best[..., None, None].repeat(3, -1), 2)[:, :, 0]
    contacts = pick(o) + pick(d) * np.where(np.isfinite(t_best), t_best, np.nan)[..., None]
    return contacts                                                                  # [G, 2, 3], NaN = no contact


def split_of(object_id):
    u = int(hashlib.md5(object_id.encode()).hexdigest(), 16) / 16 ** 32
    edges = np.cumsum(list(SPLITS.values()))
    return list(SPLITS)[int(np.searchsorted(edges, u, side="right").clip(max=len(SPLITS) - 1))]


def to_camera(camera_pose, points):
    return (points - camera_pose[:3, 3]) @ camera_pose[:3, :3]


def process_object(h5_path, seed):
    rng = np.random.default_rng(seed)
    mesh, grasps, success, mesh_file, scale = load_object(h5_path)
    idx = select_grasps(grasps, success, rng)
    grasps, success = grasps[idx], success[idx]
    object_id = os.path.splitext(os.path.basename(h5_path))[0]
    rows = []
    contacts = None
    for view in range(VIEWS_PER_OBJECT):
        obs = RENDERER.render_view(mesh, rng)
        if len(obs["points"]) < 32:
            continue
        if contacts is None:
            contacts = finger_contacts(obs["scene"], grasps)
        valid = np.isfinite(contacts).all(axis=(1, 2))
        flat = np.nan_to_num(contacts.reshape(-1, 3))
        visible = RENDERER.visible(obs["scene"], obs["eye"], flat, VISIBILITY_TOLERANCE) \
                  & DepthRenderer.in_view(obs["camera_pose"], obs["tan_limits"], flat)
        visible = visible.reshape(-1, 2) & valid[:, None]
        cls = np.where(valid, visible.sum(axis=1), -1).astype(np.int8)
        cloud_dist = cKDTree(obs["points"]).query(flat)[0].reshape(-1, 2)

        pose = obs["camera_pose"]
        cam_from_obj = np.linalg.inv(pose)
        pts = obs["points"][rng.choice(len(obs["points"]), NUM_POINTS, replace=len(obs["points"]) < NUM_POINTS)]
        rows.append({
            "object_id": object_id,
            "category": object_id.split("_")[0],
            "mesh_file": mesh_file,
            "scale": scale,
            "view": view,
            "split": split_of(object_id),
            "points": to_camera(pose, pts).astype(np.float32),
            "camera_pose": pose.astype(np.float32),
            "grasps": (cam_from_obj[None] @ grasps).astype(np.float32),
            "success": success,
            "contacts": np.where(valid[:, None, None], to_camera(pose, np.nan_to_num(contacts).reshape(-1, 3)).reshape(-1, 2, 3), np.nan).astype(np.float32),
            "contact_visible": visible.astype(np.int8),
            "contact_cloud_distance": np.where(valid[:, None], cloud_dist, np.nan).astype(np.float32),
            "visibility_class": cls,
        })
    return rows

## 4. Check one object

In [ ]:
import matplotlib.pyplot as plt

rows = process_object(h5_files[0], SEED)
r = rows[0]
print(r["object_id"], "| grasps:", len(r["grasps"]), "| class counts (-1, 0, 1, 2):",
      [int((r["visibility_class"] == c).sum()) for c in (-1, 0, 1, 2)])

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(*r["points"].T, s=1, c="0.6")
colors = {0: "tab:red", 1: "tab:orange", 2: "tab:green"}
for c, col in colors.items():
    sel = (r["visibility_class"] == c) & (r["success"] == 1)
    if sel.any():
        ax.scatter(*r["contacts"][sel].reshape(-1, 3).T, s=6, c=col, label=f"class {c}")
ax.legend(); ax.set_title("partial cloud (camera frame) and contacts of successful grasps")
plt.show()

## 5. Build the dataset

In [ ]:
from datasets import Dataset, DatasetDict, Features, Value, Sequence, Array2D, Array3D

features = Features({
    "object_id": Value("string"), "category": Value("string"), "mesh_file": Value("string"),
    "scale": Value("float32"), "view": Value("int32"), "split": Value("string"),
    "points": Array2D(shape=(None, 3), dtype="float32"),
    "camera_pose": Array2D(shape=(4, 4), dtype="float32"),
    "grasps": Array3D(shape=(None, 4, 4), dtype="float32"),
    "success": Sequence(Value("int8")),
    "contacts": Array3D(shape=(None, 2, 3), dtype="float32"),
    "contact_visible": Array2D(shape=(None, 2), dtype="int8"),
    "contact_cloud_distance": Array2D(shape=(None, 2), dtype="float32"),
    "visibility_class": Sequence(Value("int8")),
})


def generate(files):
    for path in files:
        try:
            seed = int(hashlib.md5((os.path.basename(path) + str(SEED)).encode()).hexdigest(), 16) % 2 ** 32
            yield from process_object(path, seed)
        except Exception as e:  # broken or missing meshes are skipped
            print(f"skip {os.path.basename(path)}: {e}")


full = Dataset.from_generator(generate, gen_kwargs={"files": h5_files}, features=features,
                              num_proc=NUM_PROC if len(h5_files) >= NUM_PROC > 1 else None)
dataset = DatasetDict({name: full.filter(lambda s, n=name: s == n, input_columns="split") for name in SPLITS})
dataset.save_to_disk(OUTPUT_DIR)
dataset

## 6. Statistics

In [ ]:
stats = {}
for name, ds in dataset.items():
    cls = np.concatenate([np.asarray(c) for c in ds["visibility_class"]]) if len(ds) else np.zeros(0)
    suc = np.concatenate([np.asarray(c) for c in ds["success"]]) if len(ds) else np.zeros(0)
    stats[name] = {
        "views": len(ds),
        "grasps": int(len(cls)),
        **{f"class_{c}": int((cls == c).sum()) for c in (2, 1, 0, -1)},
        **{f"class_{c}_successful": int(((cls == c) & (suc == 1)).sum()) for c in (2, 1, 0)},
    }
print(json.dumps(stats, indent=2))

## 6b. Look at samples

Pick a split and sample indices. Grasps are coloured by their visibility class
(green: both contacts visible, orange: one, red: none). Set `ONLY_SUCCESSFUL = False` to also show failed grasps.
Works on the freshly built `dataset` or on one loaded with `load_from_disk(OUTPUT_DIR)` / `load_dataset(HF_REPO_ID)`.

In [ ]:
import matplotlib.pyplot as plt
from helper.plotting import plot_grasps

VIS_SPLIT, VIS_INDICES, GRASPS_PER_CLASS, ONLY_SUCCESSFUL = "train", [0, 1, 2, 3], 10, True
CLASS_COLORS = {2: "tab:green", 1: "tab:orange", 0: "tab:red"}

ds = dataset[VIS_SPLIT].with_format("numpy")
fig = plt.figure(figsize=(4 * len(VIS_INDICES), 4))
rng = np.random.default_rng(0)
for k, idx in enumerate(VIS_INDICES):
    s = ds[int(idx)]
    sets = []
    for c, color in CLASS_COLORS.items():
        sel = np.flatnonzero((s["visibility_class"] == c) & ((s["success"] == 1) | (not ONLY_SUCCESSFUL)))
        sel = rng.choice(sel, min(GRASPS_PER_CLASS, len(sel)), replace=False) if len(sel) else sel
        sets.append((s["grasps"][sel], plt.matplotlib.colors.to_rgb(color)))
    ax = fig.add_subplot(1, len(VIS_INDICES), k + 1, projection="3d")
    counts = [int((s["visibility_class"] == c).sum()) for c in (2, 1, 0)]
    plot_grasps(ax, s["points"], sets, frame="wrist",
                title=f"{s['category']} view {s['view']}\nclasses 2/1/0: {counts[0]}/{counts[1]}/{counts[2]}")
    ax.view_init(elev=-70, azim=-90)  # roughly the camera's point of view (camera frame, z forward, y down)
plt.tight_layout(); plt.show()

## 7. Upload to the Hugging Face Hub

Needs a token with write access: set `HF_TOKEN` in the environment (Colab: *Secrets*) or log in interactively.
Set `UPLOAD = True` in the configuration and run the cell. The repository is created private unless `HF_PRIVATE = False`.

In [ ]:
CARD = f'''---
license: cc-by-nc-4.0
task_categories: [robotics]
tags: [grasping, point-cloud, partial-observation, acronym]
---
# ACRONYM partial views

Single-view, noisy partial point clouds rendered from ACRONYM objects with the Panda grasps of each object
and a visibility class per grasp (2 = both contacts visible, 1 = one, 0 = none, -1 = no contact on the mesh).
Frames: camera frame (OpenCV). `camera_pose` maps camera to mesh coordinates.
Renderer: {RENDERER.width}x{RENDERER.height} px, fov {RENDERER.fov_deg} deg, camera distance {RENDERER.distance_range} m,
axial noise {RENDERER.sigma_base} + {RENDERER.sigma_quad} z^2 m, dropout {RENDERER.dropout}, edge dropout {RENDERER.edge_dropout}.
{VIEWS_PER_OBJECT} views per object, up to {MAX_GRASPS_PER_OBJECT} grasps per object.

Statistics:
```
{json.dumps(stats, indent=2)}
```

Source data: ACRONYM (Eppner et al., ICRA 2021) and ShapeNetSem meshes. Check their licences before redistribution;
the meshes themselves are not included.
'''

if UPLOAD:
    from huggingface_hub import HfApi, login
    token = os.environ.get("HF_TOKEN")
    if token is None:
        login()  # interactive prompt
    dataset.push_to_hub(HF_REPO_ID, private=HF_PRIVATE, token=token)
    HfApi(token=token).upload_file(path_or_fileobj=CARD.encode(), path_in_repo="README.md",
                                   repo_id=HF_REPO_ID, repo_type="dataset")
    print(f"uploaded to https://huggingface.co/datasets/{HF_REPO_ID}")
else:
    print("UPLOAD is False, nothing was uploaded")